In [ ]:
print("dkfgndg")

In [1]:
%pwd
import os
os.chdir("../")
#Just went one step bcakword with this directory....

%pwd

'd:\\mlops\\Crypto_Guardian'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    local_data_files: Path 

In [ ]:
from src.Crypto.constants import *
from src.Crypto.utils.helper import read_yaml,create_directories
from src.Crypto import logger

class ConfigurationManager:
    def __init__(self,config_file_path=CONFIG_FILE_PATH,
                      params_file_path = PARAMS_FILE_PATH,
                      schema_file_path = SCHEMA_FILE_PATH):
        self.config =read_yaml( config_file_path)
        self.parmas = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)
        
        create_directories([self.config.artifacts_root])
        logger.debug(f"Till Now all the Yaml Files are Read Sucessfully...✅")
    def get_data_ingestion(self)->DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            local_data_files= config.local_data_files
        )
        logger.debug("get_data_ngestion is working compeletely fine...✅")
        return data_ingestion_config


In [ ]:
from src.Crypto import logger
import feedparser
import pandas as pd
from transformers import pipeline
import torch
from sources import rss_feeds

class DataIngestionComponent:
    def __init__(self,config: DataIngestionConfig):
        self.config = config
        
     #Now we will be Downloading all the Data for the Model Training
    def download_data(self):
        try:
            os.makedirs(os.path.dirname(self.config.local_data_files),exist_ok=True)

            # Automatically check if GPU is available
            # device=0 matlab GPU, device=-1 matlab CPU
            device = 0 if torch.cuda.is_available() else -1

            print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

            # Pipeline initialization with device
            sentiment_pipeline = pipeline("sentiment-analysis", device=device)
            logger.debug(f"Sentiment Pipeline Called and Device Decided too: {device}")

            # sentiment_pipeline = pipeline("sentiment-analysis")
            # dict ={}

            raw_data = []
            for url in rss_feeds:
                feed = feedparser.parse(url)
                for entry in feed.entries:
                    # Use .get() to avoid KeyErrors if summary is missing
                    title = entry.get('title', '')
                    summary = entry.get('summary', '')
                    full_text = f"{title} {summary}".strip()
                    
                    if full_text:
                        raw_data.append({"text": full_text})
                        
                    logger.info("List Baan rahi hai... Wait for a min...")
                    
            logger.info("List Baan gai... ✅")
            logger.debug("Batch Processing Started...✅")
            # 3. Batch process sentiments (Much faster!)
            texts = [item['text'] for item in raw_data]
            results = sentiment_pipeline(texts)
            
            logger.info("batch processign Compeleted ✅ Results are to be Merged")

            # 4. Merge results back and create DataFrame
            for i, result in enumerate(results):
                raw_data[i]['label'] = result['label']
                raw_data[i]['score'] = result['score'] # Added score for better insights
            logger.info("All Done Now Moving toward the creating the DataFrame...")
            df = pd.DataFrame(raw_data)
            
            logger.info(f" Shape of the Data: {df.shape}")

            # 5. Export
            df.to_csv(self.config.local_data_files, index=False)
            
            logger.info( f"File is Created SucessFully...✅ And saved to the {self.config.local_data_files}")
        except Exception as e:
            logger.error(f"Error in Making the File...")
            raise e

In [ ]:
#Creatign the Pipeline

try:
    cfm = ConfigurationManager()
    data_ingestion = cfm.get_data_ingestion()
    data_ingestion_component = DataIngestionComponent(data_ingestion)
    data_ingestion_component.download_data()
    logger.info("Pipeline Ran Sucessfully...✅")
except Exception as e:
    logger.error("Pipeline Error...❌")
    raise e

In [ ]:
import tqdm as notebook_tqdm

## DataBase VErsion Try karr ah hu

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    local_data_files: Path 


from src.Crypto.constants import *
from src.Crypto.utils.helper import read_yaml,create_directories
from src.Crypto import logger

class ConfigurationManager:
    def __init__(self,config_file_path=CONFIG_FILE_PATH,
                      params_file_path = PARAMS_FILE_PATH,
                      schema_file_path = SCHEMA_FILE_PATH):
        self.config =read_yaml( config_file_path)
        self.parmas = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)
        
        create_directories([self.config.artifacts_root])
        logger.debug(f"Till Now all the Yaml Files are Read Sucessfully...✅")
    def get_data_ingestion(self)->DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            local_data_files= config.local_data_files
        )
        logger.debug("get_data_ngestion is working compeletely fine...✅")
        return data_ingestion_config


from src.Crypto import logger
import feedparser
import pandas as pd
from transformers import pipeline
import torch
from sources import Links,links
import sqlite3
from googleapiclient.discovery import build
from dotenv import load_dotenv


class DataIngestionComponent:
    def __init__(self,config: DataIngestionConfig):
        self.config = config
        load_dotenv()
        self.v_ids = [i.split("v=")[1] for i in links]
        print("VIDEO IDS CONST: ",self.v_ids)
        logger.debug("Load Dot Env  and V_ids is loaded Sucessfully.....")


    def initialize_db(self):
        conn = sqlite3.connect(self.config.local_data_files)
        cursor = conn.cursor()
        
        cursor.execute('''
                       CREATE TABLE IF NOT EXISTS comments (
                           comment_id TEXT PRIMARY KEY,
                           video_id TEXT,
                            text TEXT,
                            Label TEXT,
                            Score Float32
                       )
                       ''') #This one Create the comments Table
        logger.debug("COMMENTS TABLE Create Scessfully...")
        cursor.execute(''' CREATE TABLE IF NOT EXISTS progress(
            video_id TEXT PRIMARY KEY,
            next_page_token TEXT        
            
            
            )''') # This one creates the Progress Table in the DataBase
        logger.debug("PROGReSS Table Created Sucessully.......")
        conn.commit()
        return conn
    
    def fetch_comments(self,video_id,youtube,db_conn):
        cursor = db_conn.cursor()
        
        #First We have to check whether do we have something saved Tokens to Resume???
        cursor.execute("  SELECT next_page_token FROM progress WHERE video_id = ?",(video_id,))
        #In Python, when you pass a single variable into a SQL query using a tuple, you must include a trailing comma. Without it, Python treats the parentheses as simple grouping, and sqlite3 will try to iterate over the string video_id character by character.
        
        result = cursor.fetchone()
        next_token = result[0] if result else None
        
        print("Fetching The Main Coments for: ",video_id)
        logger.info(f"Fetching The Main Coments for: {video_id}")
        
        while True:
            try:
                logger.info("Entered the Fetching Zone...")
                request = youtube.commentThreads().list(
                    part = "snippet",
                    videoId= video_id,
                    maxResults = 100,
                    pageToken = next_token,
                    fields  ="nextPageToken,items(id,snippet/topLevelComment/snippet/textDisplay)"
                    
                )
                logger.debug("Requestare Got Generated... ")
                response = request.execute()
                
                logger.debug(f"Response Generated... \n {response} ")
                #Lets Make the Batch 
                batch = []
                logger.debug("Batch Procesing IS Started...")
                print("I Am WORKING>>>>>>>>>>>>>>>>>")
                for item in response.get("items",[]):
                    
                    c_id = item['id']
                    print("C_ID: ",c_id)
                    text = item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
                    print("TEXT: ",text)
                    batch.append((c_id,video_id,text))
                    
                    
                logger.debug("Batch Processing Finished...")    
                cursor.executemany("INSERT OR IGNORE INTO comments VALUES (?,?,?)",batch)
                
                next_token = response.get("nextPageToken")
                cursor.execute("INSERT OR REPLACE INTO progress VALUES(?,?)",(video_id,next_token))
                
                db_conn.commit()
                
                if not next_token:
                    logger.debug(f"There is No New Token: {next_token}")
                    print(f"Done with video: {video_id}")
                    logger.debug("FETCHING STOPPED..........")
                    break
            
            except Exception as e:
                print(f"Error or Quota Limit: {e}")
                logger.error(f"Error or Quota Limit: {e}")
                
                # Yahan token save ho chuka hai, next time run karoge toh yahin se shuru hoga
                break
        
    def save_comments(self):
        try:
            
            api_key = os.getenv("API_KEY")
            logger.info("API_KEY Loaded Sucessfully....")
            youtube = build("youtube", "v3", developerKey=api_key)
            logger.info("Youtube Build Sucessfully....")

            db_conn = self.initialize_db()
            logger.info("DB Connection Established Sucessfully....")

            # v_ids = ["6MI0f6YjJIk"]

            for v_id in self.v_ids:
                print("VIDEO_ID: ",v_id)
                self.fetch_comments(v_id, youtube, db_conn)
                logger.info("Fetching the Comments to the Data...")

            print("All Done..........")
            db_conn.close()
        except Exception as e:
            logger.error("Something Went Wrong! ",e)
            raise e
                
                    

try:
    cfm = ConfigurationManager()
    data_ingestion = cfm.get_data_ingestion()
    data_ingestion_component = DataIngestionComponent(data_ingestion)
    data_ingestion_component.initialize_db()
    # data_ingestion_component.fetch_comments()
    data_ingestion_component.save_comments()
    logger.info("Pipeline Ran Sucessfully...✅")
except Exception as e:
    logger.error("Pipeline Error...❌")
    raise e



